In [1]:
import sys
import warnings

sys.path.append("..")
sys.dont_write_bytecode = True
warnings.filterwarnings("ignore")

In [2]:
import polars as pl
from utils.utils import (
    split_train_test,
    get_parent_run_id_from_experiment,
    compute_generalisation_error_from_run_id_and_experiment_id,
)
from utils.statics import lightgbm_model_name, FEATUERE_ENGINEERING_EXPERIMENT_ID
from utils.preprocessing_handler import PreprocessingHandler
from feature_engineering.RowWiseTransformations import RowWiseTransformations
from feature_engineering.OpenFETransformations import OpenFETransformations
from model.nested_cv_eval import ModelCVEvaluator

model_type = lightgbm_model_name

In [3]:
passes_df = pl.read_parquet("../data/02-analysis/passes.parquet")

In [4]:
train_df, test_df = split_train_test(passes_df=passes_df)

In [5]:
numerical_columns = [
    "start_x",
    "start_y",
    "end_x",
    "end_y",
    "length",
    "angle",
    "duration",
]

columns = train_df.columns
target_column = "outcome"
categorical_columns = [
    col for col in columns if col not in numerical_columns and col != target_column
]

preprocessing_handler = PreprocessingHandler(
    df=train_df, categorical_columns=categorical_columns
)
train_df_temp = preprocessing_handler.preprocess_categorical_columns()
train_df_temp = preprocessing_handler.preprocess_outcome_column()
y_train = train_df_temp.select("outcome").to_series()
X_train = preprocessing_handler.preprocess_outcome_column().drop(pl.col("outcome"))

In [ ]:
row_wise_transformations = RowWiseTransformations()
open_fe_transformations = OpenFETransformations(n_features=5)

In [7]:
categorical_columns = ["body_part", "height"]

In [11]:
run_name = f"Feature_engineering_11_OFE"
experiment_name = "Feature Engineering"
model_cv_evaluator = ModelCVEvaluator(
    model_type=model_type,
    row_wise_transformations=row_wise_transformations,
    open_fe_transformations=open_fe_transformations,
    n_inner_splits=3,
    n_outer_splits=3,
    n_trials=50,
    use_hyperparameter_tuning=False,
    use_feature_selection=False,
    use_feature_engineering=True,
    use_ofe=True,
    log_feature_importance=True,
    log_parameter_importance=True,
    run_name=run_name,
    experiment_name=experiment_name,
    categorical_columns=categorical_columns
)

In [12]:
result_cv = model_cv_evaluator.get_generalisation_error(
    X_train=X_train, y_train=y_train
)

2026-04-06 15:26:40,104 - INFO - Starting training with model lightgbm with the following configuration:
        - 3 inner splits
        - 3 outer splits
        - 50 trials
2026-04-06 15:26:40,278 - INFO - Starting full hyperparameter tuning...
2026-04-06 15:26:40,290 - INFO - Fitting final model with best hyperparameters...


2026-04-06 15:26:53,985 - INFO - Starting full hyperparameter tuning...
2026-04-06 15:26:53,985 - INFO - Fitting final model with best hyperparameters...


🏃 View run Outer_fold_1 at: http://127.0.0.1:8080/#/experiments/401199845079117708/runs/5249689c577f4936ad86777805372f81
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/401199845079117708


🏃 View run Outer_fold_2 at: http://127.0.0.1:8080/#/experiments/401199845079117708/runs/b6465da35f7d4c328d2875f149b19c92
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/401199845079117708


2026-04-06 15:27:17,348 - INFO - Starting full hyperparameter tuning...
2026-04-06 15:27:17,348 - INFO - Fitting final model with best hyperparameters...


🏃 View run Outer_fold_3 at: http://127.0.0.1:8080/#/experiments/401199845079117708/runs/526e2821f3df48bfb1ecd5722601b57a
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/401199845079117708
🏃 View run Feature_engineering_11_OFE_lightgbm at: http://127.0.0.1:8080/#/experiments/401199845079117708/runs/cfe487c6927d40ff975c612461cb3d51
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/401199845079117708


KeyboardInterrupt: 

In [ ]:
parent_run_id = get_parent_run_id_from_experiment(
    result=result_cv, experiment_id=FEATUERE_ENGINEERING_EXPERIMENT_ID
)
compute_generalisation_error_from_run_id_and_experiment_id(
    parent_run_id=parent_run_id, experiment_id=FEATUERE_ENGINEERING_EXPERIMENT_ID
)

Number of outer folds: 3
95% confidence interval for best estimate of generalisation: 0.2680379794332074 ± 0.0012790200916241532
